In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from urllib.parse import urljoin

def scrape_books(min_rating=4.0, max_price=20.0):
    """
    Scrapes book data from Books to Scrape based on minimum rating and maximum price.
    """
    base_url = "http://books.toscrape.com/"
    catalog_url = urljoin(base_url, "catalogue/")
    
    books_data = []
    
    # Mapping star rating classes to numeric values
    rating_map = {'One': 1, 'Two': 2, 'Three': 3, 'Four': 4, 'Five': 5}
    
    # Books to Scrape has 50 catalog pages in total
    for page in range(1, 51):
        page_url = f"{catalog_url}page-{page}.html"
        response = requests.get(page_url)
        if response.status_code != 200:
            break
        
        response.encoding = 'utf-8'
        soup = BeautifulSoup(response.text, 'html.parser')
        book_pods = soup.find_all('article', class_='product_pod')
        
        for pod in book_pods:
            # 1. Extract and check Rating
            rating_p = pod.find('p', class_='star-rating')
            rating_classes = rating_p.get('class', [])
            rating_str = [c for c in rating_classes if c != 'star-rating'][0]
            rating_val = rating_map.get(rating_str, 0)
            
            if rating_val < min_rating:
                continue
                
            # 2. Extract and check Price
            price_str = pod.find('p', class_='price_color').text
            price_val = float(price_str.replace('£', '').replace('Â', '').strip())
            
            if price_val > max_price:
                continue
                
            # 3. Fetch Book Detail Page if it passes preliminary filters
            book_a = pod.find('h3').find('a')
            book_relative_url = book_a['href']
            book_url = urljoin(catalog_url, book_relative_url)
            
            detail_resp = requests.get(book_url)
            if detail_resp.status_code != 200:
                continue
            detail_resp.encoding = 'utf-8'
            detail_soup = BeautifulSoup(detail_resp.text, 'html.parser')
            
            # Title
            title = detail_soup.find('h1').text.strip()
            
            # Genre from breadcrumb navigation
            breadcrumb = detail_soup.select('ul.breadcrumb li')
            genre = breadcrumb[2].text.strip() if len(breadcrumb) >= 3 else "Unknown"
            
            # Product Information Table (UPC & Availability)
            table = detail_soup.find('table', class_='table-striped')
            upc = ""
            availability = ""
            if table:
                rows = table.find_all('tr')
                for row in rows:
                    th = row.find('th').text.strip()
                    td = row.find('td').text.strip()
                    if th == 'UPC':
                        upc = td
                    elif th == 'Availability':
                        availability = td
            
            # Description
            desc_header = detail_soup.find('div', id='product_description')
            description = ""
            if desc_header and desc_header.find_next_sibling('p'):
                description = desc_header.find_next_sibling('p').text.strip()
            
            # Append collected book record
            books_data.append({
                'UPC': upc,
                'Title': title,
                'Price (£)': price_val,
                'Rating': float(rating_val),
                'Genre': genre,
                'Availability': availability,
                'Description': description
            })
            
    # Convert list of dictionaries into a Pandas DataFrame
    df = pd.DataFrame(books_data)
    return df

# Execute the function for books with a minimum rating of 4.0 and maximum price of £20
df_filtered = scrape_books(min_rating=4.0, max_price=20.0)

# Display the first few rows of the resulting DataFrame
print(df_filtered.head())

# Optional: Save the results to a CSV file
# df_filtered.to_csv('filtered_books.csv', index=False)

                UPC                                              Title  \
0  ce6396b0f23f6ecc                                        Set Me Free   
1  6258a1f6a6dcfe50  The Four Agreements: A Practical Guide to Pers...   
2  6be3beb0793a53e7                                     Sophie's World   
3  657fe5ead67a7767            Untitled Collection: Sabbath Poems 2014   
4  51653ef291ab7ddc                                    This One Summer   

   Price (£)  Rating           Genre             Availability  \
0      17.46     5.0     Young Adult  In stock (19 available)   
1      17.66     5.0    Spirituality  In stock (18 available)   
2      15.94     5.0      Philosophy  In stock (18 available)   
3      14.27     4.0          Poetry  In stock (16 available)   
4      19.49     4.0  Sequential Art  In stock (16 available)   

                                         Description  
0  Aaron Ledbetter’s future had been planned out ...  
1  In The Four Agreements, don Miguel Ruiz reveal...  